# YouTube Trending Videos: Preparation and Feature Engineering

This notebook loads the raw CSV source, cleans its fields, engineers analysis features, and writes the reproducible input for the analysis notebook.


## Unit of analysis

Each source row is a **country-day trending snapshot**, not an independent video. `clean_df` preserves those snapshots. `video_level_df` keeps the latest snapshot for each `publish_country + video_id` pair and is used where a video should count once.


In [18]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

RAW_PATH = ROOT / "data" / "raw" / "youtube_trending_raw.csv"
PROCESSED_PATH = ROOT / "data" / "processed" / "youtube_clean.csv"

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Missing raw dataset: {RAW_PATH}")

raw_df = pd.read_csv(RAW_PATH)
print(f"Loaded {RAW_PATH.relative_to(ROOT)} with {raw_df.shape[0]:,} rows and {raw_df.shape[1]} columns.")
display(raw_df.head())


Loaded data/raw/youtube_trending_raw.csv with 161,470 rows and 18 columns.


,index,video_id,trending_date,title,channel_title,category_id,publish_date,time_frame,published_day_of_week,publish_country,tags,views,likes,dislikes,comment_count,comments_disabled,ratings_disabled,video_error_or_removed
0,0,2kyS6SvSYSE,17.14.11,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,13/11/2017,17:00 to 17:59,Monday,US,SHANtell martin,748374,57527,2966,15954,False,False,False
1,1,1ZAPwfrtAFY,17.14.11,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,13/11/2017,7:00 to 7:59,Monday,US,"last week tonight trump presidency|""last week ...",2418783,97185,6146,12703,False,False,False
2,2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,12/11/2017,19:00 to 19:59,Sunday,US,"racist superman|""rudy""""|""""mancuso""""|""""king""""|""...",3191434,146033,5339,8181,False,False,False
3,3,puqaWrEC7tY,17.14.11,Nickelback Lyrics: Real or Fake?,Good Mythical Morning,24,13/11/2017,11:00 to 11:59,Monday,US,"rhett and link|""gmm""""|""""good mythical morning""...",343168,10172,666,2146,False,False,False
4,4,d380meD0W0M,17.14.11,I Dare You: GOING BALD!?,nigahiga,24,12/11/2017,18:00 to 18:59,Sunday,US,"ryan|""higa""""|""""higatv""""|""""nigahiga""""|""""i dare ...",2095731,132235,1989,17518,False,False,False


In [19]:
print("Raw schema:")
display(raw_df.dtypes.to_frame("dtype"))
print("Missing values:")
display(raw_df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("Countries:", raw_df["publish_country"].value_counts().to_dict())


Raw schema:


,dtype
index,int64
video_id,str
trending_date,str
title,str
channel_title,str
category_id,int64
publish_date,str
time_frame,str
published_day_of_week,str
publish_country,str


Missing values:


,missing_count
index,0
video_id,0
trending_date,0
title,0
channel_title,0
category_id,0
publish_date,0
time_frame,0
published_day_of_week,0
publish_country,0


Countries: {'US': 40949, 'CANADA': 40881, 'FRANCE': 40724, 'GB': 38916}


## Cleaning and engineered fields

Rates are protected against zero views. Content features describe the recorded title and tags; they do not infer a video's subject beyond its YouTube category.


In [20]:
CATEGORY_LABELS = {
    1: "Film & Animation", 2: "Autos & Vehicles", 10: "Music", 15: "Pets & Animals",
    17: "Sports", 18: "Short Movies", 19: "Travel & Events", 20: "Gaming",
    22: "People & Blogs", 23: "Comedy", 24: "Entertainment", 25: "News & Politics",
    26: "Howto & Style", 27: "Education", 28: "Science & Technology",
    29: "Nonprofits & Activism", 30: "Movies", 43: "Shows",
}
TEXT_COLUMNS = ["video_id", "title", "channel_title", "time_frame", "tags"]
NUMERIC_COLUMNS = ["category_id", "views", "likes", "dislikes", "comment_count"]
BOOLEAN_COLUMNS = ["comments_disabled", "ratings_disabled", "video_error_or_removed"]

def clean_tags(value):
    if pd.isna(value):
        return ""
    seen, tags = set(), []
    for raw_tag in str(value).split("|"):
        tag = re.sub(r"\s+", " ", raw_tag.strip().strip('"').strip("'").lower())
        if tag and tag not in {"none", "[none]", "nan", "null"} and tag not in seen:
            seen.add(tag)
            tags.append(tag)
    return "|".join(tags)

def start_hour(value):
    match = re.search(r"(\d{1,2}):\d{2}", str(value))
    return int(match.group(1)) if match else pd.NA

clean_df = raw_df.copy()
for column in TEXT_COLUMNS:
    clean_df[column] = clean_df[column].astype("string").str.strip()
clean_df["publish_country"] = clean_df["publish_country"].astype("string").str.strip().str.upper()
clean_df["published_day_of_week"] = clean_df["published_day_of_week"].astype("string").str.strip().str.title()
for column in NUMERIC_COLUMNS:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce").astype("Int64")
    if column != "category_id":
        clean_df.loc[clean_df[column] < 0, column] = pd.NA
for column in BOOLEAN_COLUMNS:
    clean_df[column] = clean_df[column].astype("string").str.lower().map({"true": True, "false": False}).astype("boolean")
clean_df["trending_date"] = pd.to_datetime(clean_df["trending_date"], format="%y.%d.%m", errors="coerce")
clean_df["publish_date"] = pd.to_datetime(clean_df["publish_date"], format="%d/%m/%Y", errors="coerce")

views = clean_df["views"].astype(float).where(clean_df["views"] > 0)
interactions = clean_df["likes"].astype(float) + clean_df["comment_count"].astype(float)
clean_df["engagement_rate"] = interactions / views
clean_df["likes_per_view"] = clean_df["likes"].astype(float) / views
clean_df["comments_per_view"] = clean_df["comment_count"].astype(float) / views
clean_df["dislikes_per_view"] = clean_df["dislikes"].astype(float) / views
clean_df["like_ratio"] = clean_df["likes"].astype(float) / (clean_df["likes"] + clean_df["dislikes"]).astype(float).where((clean_df["likes"] + clean_df["dislikes"]) > 0)
clean_df["days_to_trend"] = (clean_df["trending_date"] - clean_df["publish_date"]).dt.days.astype("Int64")
clean_df["published_hour"] = clean_df["time_frame"].map(start_hour).astype("Int64")
clean_df["publish_period"] = pd.cut(clean_df["published_hour"], [-1, 5, 11, 17, 23], labels=["Night", "Morning", "Afternoon", "Evening"])
clean_df["publish_month"] = clean_df["publish_date"].dt.month_name().str[:3]
clean_df["trending_month"] = clean_df["trending_date"].dt.to_period("M").astype("string")
clean_df["unique_video_key"] = clean_df["publish_country"].fillna("unknown") + "::" + clean_df["video_id"].fillna("unknown")
clean_df["category_label"] = clean_df["category_id"].map(CATEGORY_LABELS).fillna("Unknown")
clean_df["tags_clean"] = clean_df["tags"].map(clean_tags)
clean_df["tag_count"] = clean_df["tags_clean"].map(lambda value: 0 if not value else len(value.split("|"))).astype("Int64")
clean_df["title_length"] = clean_df["title"].str.len().astype("Int64")
clean_df["title_word_count"] = clean_df["title"].str.findall(r"\b\w+\b").str.len().astype("Int64")
clean_df["title_is_question"] = clean_df["title"].str.contains(r"\?", na=False).astype("boolean")
clean_df["title_has_exclamation"] = clean_df["title"].str.contains(r"!", na=False).astype("boolean")
clean_df["title_has_all_caps_word"] = clean_df["title"].str.contains(r"\b[A-Z]{3,}\b", na=False).astype("boolean")
clean_df["days_trending"] = clean_df.groupby("unique_video_key")["trending_date"].transform("size").astype("Int64")
bounds = clean_df.groupby("unique_video_key")["trending_date"].agg(["min", "max"])
clean_df["trend_span_days"] = clean_df["unique_video_key"].map((bounds["max"] - bounds["min"]).dt.days + 1).astype("Int64")
clean_df["market_count"] = clean_df["video_id"].map(clean_df.groupby("video_id")["publish_country"].nunique()).astype("Int64")
clean_df["metric_issue_flag"] = (clean_df[["views", "likes", "dislikes", "comment_count"]].isna().any(axis=1) | (clean_df["likes"] > clean_df["views"]) | (clean_df["comment_count"] > clean_df["views"])).astype("boolean")

assert (clean_df["days_to_trend"].dropna() >= 0).all(), "Found a negative time-to-trend value"
display(clean_df.head())


,index,video_id,trending_date,title,channel_title,category_id,publish_date,time_frame,published_day_of_week,publish_country,tags,views,likes,dislikes,comment_count,comments_disabled,ratings_disabled,video_error_or_removed,engagement_rate,likes_per_view,comments_per_view,dislikes_per_view,like_ratio,days_to_trend,published_hour,publish_period,publish_month,trending_month,unique_video_key,category_label,tags_clean,tag_count,title_length,title_word_count,title_is_question,title_has_exclamation,title_has_all_caps_word,days_trending,trend_span_days,market_count,metric_issue_flag
0,0,2kyS6SvSYSE,2017-11-14,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,22,2017-11-13,17:00 to 17:59,Monday,US,SHANtell martin,748374,57527,2966,15954,False,False,False,0.098188,0.076869,0.021318,0.003963,0.950970,1,17,Afternoon,Nov,2017-11,US::2kyS6SvSYSE,People & Blogs,shantell martin,1,34,7,False,False,True,7,7,3,False
1,1,1ZAPwfrtAFY,2017-11-14,The Trump Presidency: Last Week Tonight with J...,LastWeekTonight,24,2017-11-13,7:00 to 7:59,Monday,US,"last week tonight trump presidency|""last week ...",2418783,97185,6146,12703,False,False,False,0.045431,0.040179,0.005252,0.002541,0.940521,1,7,Morning,Nov,2017-11,US::1ZAPwfrtAFY,Entertainment,last week tonight trump presidency|last week t...,4,62,10,False,False,True,7,7,2,False
2,2,5qpjK5DgCt4,2017-11-14,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12,19:00 to 19:59,Sunday,US,"racist superman|""rudy""""|""""mancuso""""|""""king""""|""...",3191434,146033,5339,8181,False,False,False,0.048321,0.045758,0.002563,0.001673,0.964729,2,19,Evening,Nov,2017-11,US::5qpjK5DgCt4,Comedy,racist superman|rudy|mancuso|king|bach|racist|...,23,53,8,False,False,False,7,7,2,False
3,3,puqaWrEC7tY,2017-11-14,Nickelback Lyrics: Real or Fake?,Good Mythical Morning,24,2017-11-13,11:00 to 11:59,Monday,US,"rhett and link|""gmm""""|""""good mythical morning""...",343168,10172,666,2146,False,False,False,0.035895,0.029641,0.006253,0.001941,0.938550,1,11,Morning,Nov,2017-11,US::puqaWrEC7tY,Entertainment,rhett and link|gmm|good mythical morning|rhett...,27,32,5,True,False,False,7,7,2,False
4,4,d380meD0W0M,2017-11-14,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12,18:00 to 18:59,Sunday,US,"ryan|""higa""""|""""higatv""""|""""nigahiga""""|""""i dare ...",2095731,132235,1989,17518,False,False,False,0.071456,0.063097,0.008359,0.000949,0.985181,2,18,Evening,Nov,2017-11,US::d380meD0W0M,Entertainment,ryan|higa|higatv|nigahiga|i dare you|idy|rhpc|...,14,24,5,True,True,True,6,6,2,False


In [21]:
video_level_df = (clean_df.sort_values(["unique_video_key", "trending_date"], kind="mergesort")
                  .drop_duplicates("unique_video_key", keep="last").copy())
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(PROCESSED_PATH, index=False)

print(f"Saved {clean_df.shape[0]:,} cleaned snapshots to {PROCESSED_PATH.relative_to(ROOT)}")
print(f"Video-level records: {video_level_df.shape[0]:,}")
print("Date range:", clean_df["trending_date"].min().date(), "to", clean_df["trending_date"].max().date())
display(clean_df[["category_label", "engagement_rate", "days_to_trend", "days_trending", "market_count"]].describe().T)


Saved 161,470 cleaned snapshots to data/processed/youtube_clean.csv
Video-level records: 63,805
Date range: 2017-11-14 to 2018-06-14


,count,mean,std,min,25%,50%,75%,max
engagement_rate,161470.0,0.041603,0.03718,0.0,0.014351,0.031276,0.057489,0.528285
days_to_trend,161470.0,14.711271,145.192259,0.0,1.0,3.0,7.0,4215.0
days_trending,161470.0,12.992705,49.331451,1.0,2.0,4.0,12.0,530.0
market_count,161470.0,1.62014,0.892992,1.0,1.0,1.0,2.0,4.0


## Preparation conclusion

The processed CSV is the explicit handoff to notebook 2. It contains snapshot-level observations plus engineered variables; downstream sections use medians and sample thresholds because views and interactions are strongly right-skewed.
